# Chagatai Sentence Boundary Detection (Stanza Tokenizer)

This notebook trains a character-level BiLSTM tokenizer for Chagatai sentence boundary detection
using the `chagatai_only` dataset from Hugging Face Hub (`chagatai-project/chagatai-sbd`).
Configured with GPU accelerations for Kaggle 2x NVIDIA T4.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Ensure required packages are installed
for pkg in ["stanza", "datasets", "huggingface_hub"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

from huggingface_hub import login
from datasets import load_dataset
from stanza.models import tokenizer
from stanza.models.tokenization.data import TokenizationDataset
from stanza.models.tokenization.trainer import Trainer
from stanza.models.tokenization.utils import load_mwt_dict, output_predictions

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device count:", torch.cuda.device_count())
    print("Device 0:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True


In [ ]:
# 1. Hugging Face Authentication
# Tries Kaggle Secrets, environment variable, or fallback token
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face successfully.")
else:
    print("Notice: No HF_TOKEN found. If access is gated, pass token or use login().")

# 2. Load chagatai_only dataset from Hub
DATASET_CONFIG = "chagatai_only"
print(f"Loading dataset: chagatai-project/chagatai-sbd ({DATASET_CONFIG})...")
try:
    ds = load_dataset("chagatai-project/chagatai-sbd", DATASET_CONFIG, token=HF_TOKEN)
except Exception as exc:
    print(f"Hub loading error ({exc}), checking local fallback...")
    local_path = Path(f"data/UNIFIED/builds/{DATASET_CONFIG}")
    if (local_path / "train.csv").exists():
        from datasets import Dataset, DatasetDict
        ds = DatasetDict({
            "train": Dataset.from_pandas(pd.read_csv(local_path / "train.csv")),
            "validation": Dataset.from_pandas(pd.read_csv(local_path / "dev.csv")),
            "test": Dataset.from_pandas(pd.read_csv(local_path / "test.csv")),
        })
        print(f"Loaded from local fallback: {local_path}")
    else:
        raise

print("Dataset structure:", ds)


In [ ]:
WORK_DIR = Path("/kaggle/working/chagatai_sbd") if Path("/kaggle/working").exists() else Path("work/chagatai_sbd")
STANZA_DIR = WORK_DIR / "stanza"
MODEL_DIR = WORK_DIR / "models"
STANZA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def stanza_text_and_labels(tokens: list[str], word_labels: list[int]) -> tuple[str, str]:
    chars = []
    labels = []
    for token_index, (token, word_label) in enumerate(zip(tokens, word_labels)):
        if token_index:
            chars.append(" ")
            labels.append("0")
        for char_index, char in enumerate(token):
            chars.append(char)
            is_token_end = (char_index == len(token) - 1)
            if word_label and is_token_end:
                labels.append("2")
            elif is_token_end:
                labels.append("1")
            else:
                labels.append("0")
    return "".join(chars), "".join(labels)

def export_split(split_name: str, split_data) -> int:
    texts = []
    labels = []
    for row in split_data:
        tokens = row["tokens"] if isinstance(row["tokens"], list) else json.loads(row["tokens"])
        word_labels = row["labels"] if isinstance(row["labels"], list) else json.loads(row["labels"])
        text, label_text = stanza_text_and_labels(tokens, word_labels)
        assert len(text) == len(label_text), f"Length mismatch: {len(text)} vs {len(label_text)}"
        texts.append(text)
        labels.append(label_text)

    (STANZA_DIR / f"{split_name}.txt").write_text("\n\n".join(texts) + "\n\n", encoding="utf-8")
    (STANZA_DIR / f"{split_name}.toklabels").write_text("\n\n".join(labels) + "\n\n", encoding="utf-8")
    return len(texts)

dev_split = ds["validation"] if "validation" in ds else ds["dev"]
counts = {
    "train": export_split("train", ds["train"]),
    "dev": export_split("dev", dev_split),
    "test": export_split("test", ds["test"]),
}
(STANZA_DIR / "mwt.json").write_text("[]\n", encoding="utf-8")
print("Stanza data exported to:", STANZA_DIR)
print("Sample counts:", counts)


In [ ]:
# Training Configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "chg_only_tokenizer.pt"
SAVED_MODEL_PATH = MODEL_DIR / MODEL_NAME

train_args = [
    "--txt_file", str(STANZA_DIR / "train.txt"),
    "--label_file", str(STANZA_DIR / "train.toklabels"),
    "--dev_txt_file", str(STANZA_DIR / "dev.txt"),
    "--dev_label_file", str(STANZA_DIR / "dev.toklabels"),
    "--mwt_json_file", str(STANZA_DIR / "mwt.json"),
    "--lang", "chg",
    "--shorthand", "chg_only",
    "--save_dir", str(MODEL_DIR),
    "--save_name", MODEL_NAME,
    "--mode", "train",
    "--device", DEVICE,
    "--max_seqlen", "1000",
    "--batch_size", "64",
    "--lr0", "0.002",
    "--anneal", "0.999",
    "--anneal_after", "2000",
    "--dropout", "0.33",
    "--unit_dropout", "0.33",
    "--steps", "15000",
    "--eval_steps", "100",
    "--report_steps", "50",
    "--max_steps_before_stop", "1500",
    "--seed", "42",
]

print(f"Starting training on {DEVICE}...")
tokenizer.main(train_args)
print("Training finished!")


In [ ]:
# Evaluation on Chagatai Test Set
def precision_recall_f1(
    predictions: np.ndarray, gold: np.ndarray, positive_labels: set[int]
) -> tuple[float, float, float, int, int, int]:
    pred_positive = np.isin(predictions, list(positive_labels))
    gold_positive = np.isin(gold, list(positive_labels))
    tp = int(np.logical_and(pred_positive, gold_positive).sum())
    fp = int(np.logical_and(pred_positive, ~gold_positive).sum())
    fn = int(np.logical_and(~pred_positive, gold_positive).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1, tp, fp, fn

text_path = STANZA_DIR / "test.txt"
label_path = STANZA_DIR / "test.toklabels"
mwt_path = STANZA_DIR / "mwt.json"

actual_model_path = MODEL_DIR / "chg_only_nocharlm_tokenizer.pt"
if not actual_model_path.exists():
    actual_model_path = SAVED_MODEL_PATH

print(f"Evaluating model: {actual_model_path}")
runtime_args = tokenizer.parse_args([
    "--mode", "predict",
    "--txt_file", str(text_path),
    "--label_file", str(label_path),
    "--mwt_json_file", str(mwt_path),
    "--lang", "chg",
    "--shorthand", "chg_only",
    "--device", DEVICE,
    "--max_seqlen", "1000",
])

trainer = Trainer(
    args=runtime_args,
    model_file=str(actual_model_path),
    device=DEVICE,
    foundation_cache=None,
)
for key, value in trainer.args.items():
    if not key.endswith("_file") and key not in {"device", "mode", "save_dir", "load_name", "save_name"}:
        runtime_args[key] = value

batches = TokenizationDataset(
    runtime_args,
    input_files={"txt": str(text_path), "label": str(label_path)},
    vocab=trainer.vocab,
    evaluation=True,
    dictionary=trainer.dictionary,
)
mwt_dict = load_mwt_dict(str(mwt_path))
_, _, prediction_chunks, _ = output_predictions(
    None, trainer, batches, trainer.vocab, mwt_dict, runtime_args["max_seqlen"]
)

gold_chunks = batches.labels()
predictions = np.concatenate(prediction_chunks)
gold = np.concatenate(gold_chunks)

token_metrics = precision_recall_f1(predictions, gold, {1, 2, 3, 4})
sentence_metrics = precision_recall_f1(predictions, gold, {2, 4})

exact = 0
for pred_chunk, gold_chunk in zip(prediction_chunks, gold_chunks):
    pred_boundaries = np.isin(pred_chunk, [2, 4])
    gold_boundaries = np.isin(gold_chunk, [2, 4])
    exact += int(np.array_equal(pred_boundaries, gold_boundaries))

print("\n" + "=" * 50)
print("EVALUATION RESULTS ON CHAGATAI TEST SET")
print("=" * 50)
print(f"Test samples: {len(gold_chunks)}")
print(f"Token boundaries:    precision={token_metrics[0]:.4f} recall={token_metrics[1]:.4f} f1={token_metrics[2]:.4f} tp={token_metrics[3]} fp={token_metrics[4]} fn={token_metrics[5]}")
print(f"Sentence boundaries: precision={sentence_metrics[0]:.4f} recall={sentence_metrics[1]:.4f} f1={sentence_metrics[2]:.4f} tp={sentence_metrics[3]} fp={sentence_metrics[4]} fn={sentence_metrics[5]}")
print(f"Sentence exact match: {exact}/{len(gold_chunks)} = {exact / len(gold_chunks):.4f}")

metrics_record = {
    "test_samples": len(gold_chunks),
    "token_boundaries": {
        "precision": token_metrics[0], "recall": token_metrics[1], "f1": token_metrics[2],
        "tp": token_metrics[3], "fp": token_metrics[4], "fn": token_metrics[5]
    },
    "sentence_boundaries": {
        "precision": sentence_metrics[0], "recall": sentence_metrics[1], "f1": sentence_metrics[2],
        "tp": sentence_metrics[3], "fp": sentence_metrics[4], "fn": sentence_metrics[5]
    },
    "sentence_exact_match": exact / len(gold_chunks),
}
metrics_file = WORK_DIR / "chagatai_only_test_metrics.json"
metrics_file.write_text(json.dumps(metrics_record, indent=2), encoding="utf-8")
print(f"Metrics saved to: {metrics_file}")
